# WOMD v1.3.1 paper training

This notebook keeps the large Waymo files in Google Cloud, builds true-SDC causal samples, runs the preregistered four-objective/three-seed ablation, and exports only checkpoints, manifests, and metrics to Google Drive. Run the cells in order with a GPU runtime.

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

In [ ]:
import os, pathlib, subprocess, torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU before training.')
ROOT = pathlib.Path('/content/predictive-pc-fmcw')
DATA = pathlib.Path('/content/womd')
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/predictive_pc_fmcw_paper')
DATA.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

## Install the exact research code
The default URL is the paper repository. Change `REPOSITORY_URL` only if the repository was renamed.

In [ ]:
REPOSITORY_URL = 'https://github.com/panagiotagrosdouli/predictive-pc-fmcw-vehicular-communications.'
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(ROOT)], check=True)
subprocess.run(['pip', 'install', '-q', '-e', f'{ROOT}[ml]'], check=True)
subprocess.run(['pip', 'install', '-q', 'waymo-open-dataset-tf-2-12-0'], check=True)

## Select a paper-scale subset
Training and validation shards remain separate. Start with `SMOKE=True`; after it passes, set it to `False`. The full setting uses 50 training shards. Official validation shards are downloaded separately and preserved for the held-out phase.

In [ ]:
SMOKE = True
TRAIN_SHARDS = 1 if SMOKE else 50
VALIDATION_SHARDS = 1 if SMOKE else 40
BUCKET = 'gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario'
train_dir = DATA / 'training'
validation_dir = DATA / 'validation'
train_dir.mkdir(exist_ok=True)
validation_dir.mkdir(exist_ok=True)
for i in range(TRAIN_SHARDS):
    name = f'training.tfrecord-{i:05d}-of-01000'
    target = train_dir / name
    if not target.exists():
        subprocess.run(['gcloud', 'storage', 'cp', f'{BUCKET}/training/{name}', str(target)], check=True)
for i in range(VALIDATION_SHARDS):
    name = f'validation.tfrecord-{i:05d}-of-00150'
    target = validation_dir / name
    if not target.exists():
        subprocess.run(['gcloud', 'storage', 'cp', f'{BUCKET}/validation/{name}', str(target)], check=True)
print('Downloaded', len(list(train_dir.iterdir())), 'training and', len(list(validation_dir.iterdir())), 'validation shards')

## Build causal true-SDC samples
Only training shards enter model fitting. Official validation remains isolated from this command.

In [ ]:
train_npz = DATA / 'womd_training_samples.npz'
cmd = ['python', str(ROOT/'scripts/01_build_official_womd_samples.py'), *map(str, sorted(train_dir.iterdir())), '--output', str(train_npz), '--max-vehicles', '16']
if SMOKE:
    cmd += ['--max-scenarios', '200']
subprocess.run(cmd, cwd=ROOT, check=True)

## Validate the immutable experiment plan
The publication run is four objectives times three independent seeds.

In [ ]:
seeds = ['20260827', '20260828', '20260829']
epochs = '2' if SMOKE else '80'
subprocess.run(['python', str(ROOT/'scripts/04_run_training_ablation.py'), str(train_npz), '--output', str(DRIVE_OUT/'learned_ablation'), '--epochs', epochs, '--seeds', *seeds, '--plan-only'], cwd=ROOT, check=True)

## Run training
Every completed run is written directly to Drive. If Colab disconnects, keep the existing outputs and rerun only missing objective/seed directories.

In [ ]:
subprocess.run(['python', str(ROOT/'scripts/04_run_training_ablation.py'), str(train_npz), '--output', str(DRIVE_OUT/'learned_ablation'), '--epochs', epochs, '--batch-size', '64', '--seeds', *seeds, '--lambda-link', '0.2', '--lambda-outage', '0.1'], cwd=ROOT, check=True)

## Export small artifacts
Do not package TFRecords. The archive contains only checkpoints, plans, configurations, and metrics.

In [ ]:
archive = DRIVE_OUT / 'womd_learned_paper_artifacts.tar.gz'
subprocess.run(['tar', '-czf', str(archive), '-C', str(DRIVE_OUT), 'learned_ablation'], check=True)
print('Created:', archive)